# IWI Employee Contacts

This Jupyter notebook scrapes the ZHAW-IWI employee overview page, follows links to individual profiles, and creates a DataFrame with last name, first name, email, and phone number.

Source: https://www.zhaw.ch/de/sml/institute-zentren/iwi/ueber-uns

## Libraries and Settings

In [21]:
import json
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://www.zhaw.ch"
OVERVIEW_URL = "https://www.zhaw.ch/de/sml/institute-zentren/iwi/ueber-uns"

# Headers to mimic a browser request
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "de-CH,de;q=0.9",
}

## Step 1: Load Overview Page and Extract Profile Links

In [ ]:
response = requests.get(OVERVIEW_URL, headers=HEADERS, timeout=15)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# Collect all links to person profiles (pattern: /de/ueber-uns/person/...)
profile_links = []
seen = set()
for a in soup.find_all("a", href=True):
    href = a["href"]
    if re.match(r"^/de/ueber-uns/person/", href) and href not in seen:
        seen.add(href)
        profile_links.append(BASE_URL + href)

print(f"Found {len(profile_links)} profile links")
profile_links[:5]

48 Profil-Links gefunden


['https://www.zhaw.ch/de/ueber-uns/person/shhd',
 'https://www.zhaw.ch/de/ueber-uns/person/grsu',
 'https://www.zhaw.ch/de/ueber-uns/person/kizi',
 'https://www.zhaw.ch/de/ueber-uns/person/obel',
 'https://www.zhaw.ch/de/ueber-uns/person/gues']

## Step 2: Scrape Individual Profile Pages

In [ ]:
# Title prefixes used at ZHAW - stripped to isolate the actual name
TITLE_PREFIXES = [
    "Prof. Dr. sc. nat. ETH", "Prof. Dr. med.", "Prof. Dr. phil.",
    "Prof. Dr. rer. nat.", "Prof. Dr.", "Prof.",
    "Dr. sc. nat. ETH", "Dr. rer. nat.", "Dr. med.", "Dr. phil.",
    "Dr. sc.", "Dr.",
    "Dipl.-Ing.", "Dipl.-Phys.", "Dipl.-Inf.",
    "lic. iur.", "MLaw", "MBA", "MAS", "CAS",
]

def strip_titles(full_name: str) -> str:
    """Remove academic title prefixes from a display name."""
    name = full_name.strip()
    changed = True
    while changed:
        changed = False
        for prefix in TITLE_PREFIXES:
            if name.startswith(prefix + " "):
                name = name[len(prefix):].strip()
                changed = True
    return name


def parse_name(full_display_name: str) -> tuple[str, str]:
    """Return (first_name, last_name) from a full display name.

    Strategy:
    1. Try JSON-LD structured data on the page (most reliable).
    2. Fall back to stripping titles and splitting on the last word.
    """
    clean = strip_titles(full_display_name)
    # Also strip trailing suffixes like ", MBA"
    clean = re.sub(r",\s*\S+$", "", clean).strip()
    parts = clean.split()
    if len(parts) >= 2:
        vorname = " ".join(parts[:-1])
        nachname = parts[-1]
    elif len(parts) == 1:
        vorname = ""
        nachname = parts[0]
    else:
        vorname = ""
        nachname = full_display_name
    return vorname, nachname


def scrape_profile(url: str) -> dict:
    """Fetch a ZHAW person profile page and return contact fields."""
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    ps = BeautifulSoup(resp.text, "html.parser")

    # --- Try JSON-LD structured data first (provides givenName / familyName) ---
    given_name, family_name = None, None
    for script in ps.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string)
            if isinstance(data, list):
                data = next((d for d in data if d.get("@type") == "Person"), {})
            if data.get("@type") == "Person":
                given_name = data.get("givenName", "").strip() or None
                family_name = data.get("familyName", "").strip() or None
                break
        except (json.JSONDecodeError, AttributeError):
            pass

    # --- Full display name from <h1> ---
    h1 = ps.find("h1")
    full_name = h1.get_text(strip=True) if h1 else ""

    if not given_name or not family_name:
        given_name_fb, family_name_fb = parse_name(full_name)
        given_name = given_name or given_name_fb
        family_name = family_name or family_name_fb

    # --- Phone: look for <a href="tel:..."> ---
    tel_tag = ps.find("a", href=re.compile(r"^tel:"))
    if tel_tag:
        tel_raw = tel_tag["href"].replace("tel:", "")
        # Format example: +41589347002 -> +41 (0) 58 934 70 02
        tel = tel_tag.get_text(strip=True) or tel_raw
    else:
        tel = ""

    # --- Email: look for <a href="mailto:..."> or text containing @zhaw.ch ---
    email = ""
    mail_tag = ps.find("a", href=re.compile(r"^mailto:"))
    if mail_tag:
        email = mail_tag["href"].replace("mailto:", "").strip()
    else:
        # Fallback: search all text nodes for @zhaw.ch pattern
        match = re.search(r"[\w.+-]+@zhaw\.ch", ps.get_text())
        if match:
            email = match.group(0)

    return {
        "Vorname": given_name,
        "Name": family_name,
        "Email": email,
        "Telefon": tel,
        "Profil_URL": url,
    }

In [ ]:
records = []
for i, url in enumerate(profile_links):
    try:
        record = scrape_profile(url)
        records.append(record)
        print(f"[{i+1:02d}/{len(profile_links)}] OK  {record['Vorname']} {record['Name']}")
    except Exception as e:
        print(f"[{i+1:02d}/{len(profile_links)}] ERR {url}  -> {e}")
    time.sleep(0.5)  # polite delay between requests

print(f"\nSuccessfully scraped: {len(records)} out of {len(profile_links)} profiles")

[01/48] OK  Theresa Schmiedel
[02/48] OK  Jasmin Grassmugg
[03/48] OK  Anette Kizilelma
[04/48] OK  Dominique Oberle
[05/48] OK  Andrea Maria Günster
[06/48] OK  Christian Russ
[07/48] OK  Christian Hitz
[08/48] OK  Mario Gellrich
[09/48] OK  Pasquale Cirillo
[10/48] OK  Maria Rothstein
[11/48] OK  Václav Pechtor
[12/48] OK  Alexander Walter
[13/48] OK  Sylvain Rossi
[14/48] OK  Dominik Meier
[15/48] OK  Fabian Oehninger
[16/48] OK  Nico Ebert
[17/48] OK  Benjamin Ambühl
[18/48] OK  Thierry Schaltegger
[19/48] OK  Christian Weber
[20/48] OK  Tibor Dudas
[21/48] OK  Stefan Kemper
[22/48] OK  Tim Geppert
[23/48] OK  Björn Scheppler
[24/48] OK  Mohamed Tahar Kerkeni
[25/48] OK  Andreas Block
[26/48] OK  Ninja Leikert-Böhm
[27/48] OK  Katja Kurz
[28/48] OK  Philipp Matter
[29/48] OK  Gianluca Miscione
[30/48] OK  Tibor Pimentel
[31/48] OK  Marcel Sieber
[32/48] OK  Alexandre de Spindler
[33/48] OK  Elena Gavagnin
[34/48] OK  Max Meisterhans
[35/48] OK  Adrian Moser
[36/48] OK  David Grüner

## Step 3: Create and Display DataFrame

In [25]:
df = pd.DataFrame(records, columns=["Vorname", "Name", "Email", "Telefon", "Profil_URL"])
df = df.sort_values(["Name", "Vorname"]).reset_index(drop=True)

print(f"Shape: {df.shape}")
df

Shape: (48, 5)


,Vorname,Name,Email,Telefon,Profil_URL
0,Benjamin,Ambühl,benjamin.ambuehl@zhaw.ch,+41 (0) 58 934 43 12,https://www.zhaw.ch/de/ueber-uns/person/ambe
1,Andreas,Block,andreas.block@zhaw.ch,+41 (0) 58 934 45 90,https://www.zhaw.ch/de/ueber-uns/person/bloc
2,Elke,Brucker-Kley,elke.brucker-kley@zhaw.ch,+41 (0) 58 934 66 85,https://www.zhaw.ch/de/ueber-uns/person/brck
3,Pasquale,Cirillo,pasquale.cirillo@zhaw.ch,+41 (0) 58 934 44 36,https://www.zhaw.ch/de/ueber-uns/person/ciri
4,Tibor,Dudas,tibor.dudas@zhaw.ch,+41 (0) 58 934 42 18,https://www.zhaw.ch/de/ueber-uns/person/duda
5,Nico,Ebert,nico.ebert@zhaw.ch,+41 (0) 58 934 46 75,https://www.zhaw.ch/de/ueber-uns/person/ebet
6,Katja,Fraedrich,katja.fraedrich@zhaw.ch,+41 (0) 58 934 73 09,https://www.zhaw.ch/de/ueber-uns/person/fraj
7,Elena,Gavagnin,elena.gavagnin@zhaw.ch,+41 (0) 58 934 46 12,https://www.zhaw.ch/de/ueber-uns/person/gava
8,Mario,Gellrich,mario.gellrich@zhaw.ch,+41 (0) 58 934 78 90,https://www.zhaw.ch/de/ueber-uns/person/gell
9,Tim,Geppert,tim.geppert@zhaw.ch,+41 (0) 58 934 48 22,https://www.zhaw.ch/de/ueber-uns/person/gepp


In [ ]:
# Quick overview: missing values
print("Missing values per column:")
print(df.replace("", pd.NA).isna().sum())

Fehlende Werte pro Spalte:
Vorname       0
Name          0
Email         0
Telefon       0
Profil_URL    0
dtype: int64


### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [27]:
import os
import platform
from datetime import datetime
from platform import python_version

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1044-azure
Datetime: 2026-03-05 16:06:00
Python Version: 3.11.14
-----------------------------------
